In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('sales_messy.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    str    
 2   customer_id  200 non-null    float64
 3   country      208 non-null    str    
 4   category     208 non-null    str    
 5   product      208 non-null    str    
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 14.8 KB


In [5]:
print("Before:", df.shape)

Before: (208, 9)


In [6]:
df = df.drop_duplicates()

In [7]:
print("After:", df.shape)

After: (200, 9)


In [8]:
df['discount'] = df['discount'].fillna(0)

In [9]:
df['unit_price'] = df['unit_price'].fillna(df['unit_price'].median())

In [10]:
print(df[['discount', 'unit_price']].isnull().sum())

discount      0
unit_price    0
dtype: int64


In [11]:
before = df.shape[0]

In [12]:
df = df.dropna(subset=['customer_id'])

In [13]:
print("Dropped:", before - df.shape[0])

Dropped: 7


In [14]:
df['customer_id'] = df['customer_id'].astype(int)

In [15]:
df['order_date'] = pd.to_datetime(df['order_date'])

In [16]:
df.info()

<class 'pandas.DataFrame'>
Index: 193 entries, 1 to 199
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     193 non-null    int64         
 1   order_date   193 non-null    datetime64[us]
 2   customer_id  193 non-null    int64         
 3   country      193 non-null    str           
 4   category     193 non-null    str           
 5   product      193 non-null    str           
 6   quantity     193 non-null    int64         
 7   unit_price   193 non-null    float64       
 8   discount     193 non-null    float64       
dtypes: datetime64[us](1), float64(2), int64(3), str(3)
memory usage: 15.1 KB


In [17]:
df['revenue'] = df['quantity'] * df['unit_price'] * (1 - df['discount'])

In [18]:
df['month'] = df['order_date'].dt.month

In [19]:
customers = pd.read_csv('customers.csv')

In [20]:
before = df.shape[0]

In [21]:
df = df.merge(customers, on='customer_id', how='left')

In [22]:
print(before, df.shape[0])

193 193


In [23]:
total = df['revenue'].sum()


In [24]:
df.groupby('category')['revenue'].sum().sort_values(ascending=False)

category
Laptops        161187.4000
Phones          63403.4000
Monitors        58295.5500
Accessories     10323.5465
Name: revenue, dtype: float64

In [25]:
df.groupby('month')['revenue'].sum()

month
1     15348.3950
2     19631.0780
3     19836.5880
4     26456.2405
5     23633.5065
6     24754.8375
7     42529.4260
8     30827.4315
9     14637.5020
10    33697.7500
11    19117.3905
12    22739.7510
Name: revenue, dtype: float64

In [26]:
df.groupby('segment').agg(
    total_revenue=('revenue', 'sum'),
    orders=('order_id', 'count'),
    avg_order=('revenue', 'mean'),
).round(2).sort_values('total_revenue', ascending=False)

,total_revenue,orders,avg_order
segment,,,
Consumer,174850.84,106,1649.54
Education,77782.03,55,1414.22
Business,40577.03,32,1268.03


In [27]:
    top_cat    = df.groupby('category')['revenue'].sum().idxmax()
    top_share  = round(df.groupby('category')['revenue'].sum().max() / total * 100, 1)
    best_month = df.groupby('month')['revenue'].sum().idxmax()
    top_seg    = df.groupby('segment')['revenue'].sum().idxmax()
    
    print(f'1. Top category : {top_cat} — {top_share}% of total revenue')
    print(f'2. Best month   : {best_month}')
    print(f'3. Top segment  : {top_seg}')

1. Top category : Laptops — 55.0% of total revenue
2. Best month   : 7
3. Top segment  : Consumer


In [28]:
print(df.isnull().sum())

order_id         0
order_date       0
customer_id      0
country_x        0
category         0
product          0
quantity         0
unit_price       0
discount         0
revenue          0
month            0
customer_name    0
country_y        0
signup_date      0
segment          0
dtype: int64
